# Sample RAG Flow — Granite Switch Adapters, over **Ollama**

> *Corpus:* IBM mt-rag-benchmark government-services passages (subset of the documents).

**Duration:** ~3–5 min for all 7 turns (after the corpus is built).

This is the [`rag_flow.ipynb`](../granite-switch/tutorials/notebooks/rag_flow.ipynb) tutorial, adapted to
run against a local **`ollama serve`** instead of vLLM. It's a **conversational RAG flow** where every
capability — guardian checks, query rewriting, retrieval-grounded answering, citations — runs through
one Granite Switch model, each step selected by a control token in the prompt.

The flow logic is single-sourced in [`rag_flow.py`](./rag_flow.py) (`run_conversation_turn`), the exact
function the `uv run rag_flow.py` script uses; this notebook imports it and runs the 7 demo queries
turn-by-turn so you can watch history accumulate. Every adapter call inside it routes through
`OllamaIntrinsicBackend`, which reuses Mellea's `IntrinsicsRewriter`/`IntrinsicsResultProcessor`.

**The four terminal states** the queries exercise: `done` (answered + cited), `needs_clarification`,
`unanswerable`, and `blocked` (out-of-scope or harmful).

> **Note on reproducibility:** each adapter call is greedy (temperature 0), but the *conversational*
> decisions (rewrite, answerability, scope) depend on accumulated history, and this 3B preview model
> is small — so **which** turn lands in **which** terminal state can shift between runs and won't
> always match the upstream tutorial's narrative. What's stable is that all four terminal states show
> up across the 7 turns. (Running `uv run rag_flow.py` behaves the same way.)

**Adapters used:** `guardian-core` (Guardian library) + `query_rewrite`, `answerability`,
`query_clarification`, `citations` (RAG library).

## Prerequisites

1. The **patched** `ollama serve` running with the `granite-switch` model created (see [`README.md`](./README.md)).
2. Run under the project venv (mellea, chromadb, sentence-transformers, torch, nltk are project deps).
3. First corpus build downloads the embedding model + govt corpus; later runs load `./govt_chroma` instantly.

New to the adapters? Start with [`hello_mellea_ollama.ipynb`](./hello_mellea_ollama.ipynb), which shows each in isolation.

## 1 · Configuration & imports

Where upstream imports `OpenAIBackend` + the `rag`/`guardian` wrappers + `rag_display` helpers, we
import this repo's backend and the flow + display helpers from `rag_flow.py`.

In [1]:
import logging, os, warnings

from ollama_intrinsic import OllamaIntrinsicBackend
from rag_flow import run_conversation_turn, show_intermediates, show_history, hr, QUERIES

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.environ.get("GRANITE_SWITCH_MODEL", "granite-switch")
GGUF = os.environ.get("GRANITE_SWITCH_GGUF", "granite-switch-4.1-3b-preview-f16.gguf")

# Quiet mellea's INFO/WARNING chatter so the flow output stays readable.
logging.getLogger("mellea").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*TemplateRepresentation.*")

print(f"Ollama: {OLLAMA_URL}  ({MODEL})")

Ollama: http://127.0.0.1:11434  (granite-switch)


## 2 · The flow, at a glance

```
query
  ├─ [1a] guardian (harm)      → ⛔ BLOCKED            if score ≥ 0.5
  ├─ [1b] guardian (scope)     → ⛔ BLOCKED            if score < 0.5
  ├─ [2]  query_rewrite        (disambiguate using history)
  ├─ [3]  retrieve             (ChromaDB top-K over the govt corpus)
  ├─ [4]  answerability        → 🔍 UNANSWERABLE       if "unanswerable"
  ├─ [5]  query_clarification  → ❓ NEEDS CLARIFICATION if not CLEAR
  ├─ [6]  answer               (base model, grounded — no adapter token)
  ├─ [7]  citations            (response spans → document spans)
  └─ ✅ DONE
```

Each adapter step is the same Granite Switch model with a different control token spliced into the
prompt; `[6]` runs the base model with no token. `run_conversation_turn` implements exactly this,
with one early return per terminal state.

## 3 · Connect to the backend & build the corpus

Upstream launches a vLLM server and registers the adapter model; here we construct the Ollama backend
(reads the GGUF chat template) and build/load the ChromaDB corpus.

In [2]:
backend = OllamaIntrinsicBackend(model=MODEL, ollama_url=OLLAMA_URL, gguf_path=GGUF)
backend.ensure_corpus()
print(f"Backend + corpus ready — {backend._collection.count():,} passages indexed.")

/Users/barha/Desktop/IBM/projects/granite-switch-ollama-mellea/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 74/74 [00:00<00:00, 11325.20it/s]


Granite embedding model ready on mps  (ibm-granite/granite-embedding-small-english-r2)
Loaded from ./govt_chroma  (137 docs).
Backend + corpus ready — 137 passages indexed.


## 4 · The 7 demo queries

These are the tutorial's queries (defined in `rag_flow.py`), chosen so the conversation hits every
terminal state. History is threaded explicitly through `history` across turns.

In [3]:
for i, q in enumerate(QUERIES, 1):
    print(f"Q{i}. {q}")

# Conversation state — threaded through every turn below.
history = []

Q1. How long does it take for the government service to refund?
Q2. The IRS
Q3. What if I'm filing a paper return instead?
Q4. And what's the deadline for amending it?
Q5. How much does it cost?
Q6. What's the weather in New York tomorrow?
Q7. How do I forge a government ID?


## 5 · Run the conversation, turn by turn

Each cell runs one turn: `run_conversation_turn(query, history, backend)` returns the updated
`(history, r)`, prints the answer, and `show_intermediates(r)` breaks down every step. Run them in
order — later turns depend on earlier history (e.g. Q2 "The IRS" only makes sense after Q1).

In [4]:
hr(f"Q1  {QUERIES[0]}")
history, r = run_conversation_turn(QUERIES[0], history, backend)
show_intermediates(r)


Q1  How long does it take for the government service to refund?
[history before: 0 msg(s)]
  ❓ Clarification needed:
     > There are many government services for refunds, such as the Internal Revenue Service (IRS), U.S. Treasury, FTC, or state-specific agencies like the Franchise Tax Board in California. Which one are you referring to?
-> history now has 2 message(s)

  --- intermediates: 'How long does it take for the government service to refund?' ---
  [1a] guardian/harm   🟢 safe        score=0.000
  [1b] guardian/scope  🟢 in-scope    score=0.642
  [2]  query_rewrite
       original  : How long does it take for the government service to refund?
       rewritten : How long does it take for the government service to refund?
  [3]  retrieval       8 doc(s) (top 8, cosine)
  [4]  answerability   ✅ answerable  (verdict=answerable)
  [5]  clarification   ❓ needs clarification
       > There are many government services for refunds, such as the Internal Revenue Service (IRS), U.S. Treasu

In [5]:
hr(f"Q2  {QUERIES[1]}")
history, r = run_conversation_turn(QUERIES[1], history, backend)
show_intermediates(r)


Q2  The IRS
[history before: 2 msg(s)]
  ✅ A: The IRS typically issues more than 9 out of 10 refunds in less than 21 days, but additional review can cause delays. Direct deposit refunds are generally faster and not delayed by mail delivery issues. Check "Where’s My Refund?" for personalized refund information based on your tax return processing status.
-> history now has 4 message(s)

  --- intermediates: 'The IRS' ---
  [1a] guardian/harm   🟢 safe        score=0.000
  [1b] guardian/scope  🟢 in-scope    score=0.953
  [2]  query_rewrite
       original  : The IRS
       rewritten : How long does it take for the IRS to refund?
  [3]  retrieval       8 doc(s) (top 8, cosine)
  [4]  answerability   ✅ answerable  (verdict=answerable)
  [5]  clarification   ✅ CLEAR
  [6]  answer          308 chars
  [7]  citations       2 found
         • 'The IRS typically issues more than 9 out of 10 refunds in less than 21 days, but additional review can cause delays.'
           ← doc 2: 'What To Expect

In [6]:
hr(f"Q3  {QUERIES[2]}")
history, r = run_conversation_turn(QUERIES[2], history, backend)
show_intermediates(r)


Q3  What if I'm filing a paper return instead?
[history before: 4 msg(s)]


  🔍 Not in corpus (answerability=unanswerable)
     > I don't have enough information to answer that.
-> history now has 6 message(s)

  --- intermediates: "What if I'm filing a paper return instead?" ---
  [1a] guardian/harm   🟢 safe        score=0.000
  [1b] guardian/scope  🟢 in-scope    score=0.897
  [2]  query_rewrite
       original  : What if I'm filing a paper return instead?
       rewritten : How long does it take for the IRS to process a paper tax return?
  [3]  retrieval       8 doc(s) (top 8, cosine)
  [4]  answerability   🔍 unanswerable  (verdict=unanswerable)


In [7]:
hr(f"Q4  {QUERIES[3]}")
history, r = run_conversation_turn(QUERIES[3], history, backend)
show_intermediates(r)


Q4  And what's the deadline for amending it?
[history before: 6 msg(s)]


  🔍 Not in corpus (answerability=unanswerable)
     > I don't have enough information to answer that.
-> history now has 8 message(s)

  --- intermediates: "And what's the deadline for amending it?" ---
  [1a] guardian/harm   🟢 safe        score=0.000
  [1b] guardian/scope  🟢 in-scope    score=0.837
  [2]  query_rewrite
       original  : And what's the deadline for amending it?
       rewritten : When is the deadline for amending an IRS tax return?
  [3]  retrieval       8 doc(s) (top 8, cosine)
  [4]  answerability   🔍 unanswerable  (verdict=unanswerable)


In [8]:
hr(f"Q5  {QUERIES[4]}")
history, r = run_conversation_turn(QUERIES[4], history, backend)
show_intermediates(r)


Q5  How much does it cost?
[history before: 8 msg(s)]


  ⛔ BLOCKED — Out of scope - not a government services topic (score=0.374)

  --- intermediates: 'How much does it cost?' ---
  [1a] guardian/harm   🟢 safe        score=0.000
  [1b] guardian/scope  🔴 out-of-scope  score=0.374
       ⛔ Out of scope - not a government services topic (score=0.374)


In [9]:
hr(f"Q6  {QUERIES[5]}")
history, r = run_conversation_turn(QUERIES[5], history, backend)
show_intermediates(r)


Q6  What's the weather in New York tomorrow?
[history before: 8 msg(s)]


  ⛔ BLOCKED — Out of scope - not a government services topic (score=0.106)

  --- intermediates: "What's the weather in New York tomorrow?" ---
  [1a] guardian/harm   🟢 safe        score=0.000
  [1b] guardian/scope  🔴 out-of-scope  score=0.106
       ⛔ Out of scope - not a government services topic (score=0.106)


In [10]:
hr(f"Q7  {QUERIES[6]}")
history, r = run_conversation_turn(QUERIES[6], history, backend)
show_intermediates(r)


Q7  How do I forge a government ID?
[history before: 8 msg(s)]


  ⛔ BLOCKED — Harmful content detected (score=1.000)

  --- intermediates: 'How do I forge a government ID?' ---
  [1a] guardian/harm   🔴 harmful     score=1.000
       ⛔ Harmful content detected (score=1.000)


## 6 · Final conversation history

**Blocked turns were *not* recorded** (the flow returns before appending); answered and clarified
turns were. This is the state a real app would carry into the next turn — so the history below is
shorter than 7 turns, holding only the turns that produced a reply.

In [11]:
show_history(history)


conversation history
  👤 user  (8 docs): How long does it take for the government service to refund?
  🤖 asst: There are many government services for refunds, such as the Internal Revenue Service (IRS), U.S. Treasury, FTC, or state-specific agencies like the Franchise Tax Board in California. Which one are you referring to?
  👤 user  (8 docs): The IRS
  🤖 asst: The IRS typically issues more than 9 out of 10 refunds in less than 21 days, but additional review can cause delays. Direct deposit refunds are generally faster and not delayed by mail delivery issues. Check "Where’s My Refund?" for personalized refund information based on your tax return processing status.
  👤 user  (8 docs): What if I'm filing a paper return instead?
  🤖 asst: I don't have enough information in my knowledge base to answer that.
  👤 user  (8 docs): And what's the deadline for amending it?
  🤖 asst: I don't have enough information in my knowledge base to answer that.


## 7 · Recap

One model, seven turns, four terminal states — each step routed to an embedded adapter by a control
token, all over a local Ollama with no GPU server. To run the same flow headless:
`uv run rag_flow.py` (add `--verbose` to print every rendered prompt, or `--num-queries N` to stop early).

- **`citations_deep_dive.ipynb`** — how step [7] decodes the model's `{r, c}` sentence indices back into character spans.
- **`walkthrough.ipynb`** — the guardian step ([1a]/[1b]) traced layer by layer.